In [ ]:
import numpy as np

# --------------------------------------------------------
# 1. 하드웨어 파라미터 설정
# --------------------------------------------------------
M, N = 8, 8         # 8x8 MAC Array (출력 크기)
K = 8              # 연속 누산 횟수 (Cycle)
SHIFT_AMT = 8       # Truncation: 하위 8비트 버림

# 시드(Seed)를 고정하여 항상 동일한 난수가 나오게 설정 (하드웨어 디버깅 필수)
np.random.seed(42)

# --------------------------------------------------------
# 2. 난수 생성 (Activation & Weight: -128 ~ 127)
# --------------------------------------------------------
# np.random.randint는 high 값을 포함하지 않으므로 128로 설정
A_act = np.random.randint(-128, 128, size=(M, K), dtype=np.int8)
W_wt   = np.random.randint(-128, 128, size=(K, N), dtype=np.int8)

print("[Step 1] 8-bit Signed 난수 행렬 생성 완료 (A: 8x8, W: 8x8)")

# --------------------------------------------------------
# 3. 32-bit 누산기 행렬 곱 (Golden Matmul)
# --------------------------------------------------------
# NumPy 연산 시 8-bit끼리 곱하면 SW에서도 오버플로우가 나므로,
# 하드웨어의 32-bit 누산기와 동일하게 동작하도록 미리 int32로 캐스팅 후 연산
acc_32bit = np.matmul(A_act.astype(np.int32), W_wt.astype(np.int32))

# --------------------------------------------------------
# 4. Truncation (버림) 및 Saturation (포화) 적용
# --------------------------------------------------------
# 파이썬의 >> 연산자는 음수일 때 부호 비트를 유지하는 산술 시프트(Arithmetic Shift)를
# 수행하므로 Verilog의 $signed() 시프트 연산과 정확히 100% 동일하게 동작합니다.
scaled_result = acc_32bit >> SHIFT_AMT

# 8-bit 표현 범위(-128 ~ 127)를 넘어가는 값을 잘라내는 포화(Saturation) 연산
golden_out_8bit = np.clip(scaled_result, -128, 127).astype(np.int8)

print("[Step 2] 32-bit 누산 -> Truncation -> Saturation 적용 완료")

# --------------------------------------------------------
# 5. Verilog Testbench용 Hex 파일 추출 (File I/O)
# --------------------------------------------------------
def save_hex_file(filename, data_array, bit_width):
    """Verilog의 $readmemh가 읽을 수 있는 2의 보수 Hex 포맷으로 저장"""
    mask = (1 << bit_width) - 1
    hex_len = bit_width // 4

    with open(filename, 'w') as f:
        # 1D 배열로 쭉 펴서(flatten) 한 줄에 하나씩 저장
        for val in data_array.flatten().tolist():
            # 255 마스킹 연산 작동
            hex_str = f"{(val & mask):0{hex_len}X}\n"
            f.write(hex_str)
    print(f"  -> '{filename}' 저장 완료")

print("\n[Step 3] Verilog 연동용 Hex 파일 생성 중...")
save_hex_file('activation_in.txt', A_act, 8)
save_hex_file('weight_in.txt', W_wt, 8)
save_hex_file('golden_acc32.txt', acc_32bit, 32)
save_hex_file('golden_out_8bit.txt', golden_out_8bit, 8)

[Step 1] 8-bit Signed 난수 행렬 생성 완료 (A: 8x8, W: 8x8)
[Step 2] 32-bit 누산 -> Truncation -> Saturation 적용 완료

[Step 3] Verilog 연동용 Hex 파일 생성 중...
  -> 'activation_in.txt' 저장 완료
  -> 'weight_in.txt' 저장 완료
  -> 'golden_acc32.txt' 저장 완료
  -> 'golden_out_8bit.txt' 저장 완료
